# Deep Learning  Framework for Building Change Detection From High-Resolution Remote Sensing Images

## Phase 1: High-End Data Engineering
This notebook implements a "physics-aware" data pipeline for Urban Change Detection.
It covers:
1. **Triplet Mapping**: $t_0$, $t_1$, and Label.
2. **Intelligent Tiling**: 1024x1024 $\to$ 256x256 with Change-Density Filtering.
3. **Difference Feature Mapping**: Explicitly providing $|B-A|$ to the model.
4. **Synchronized Augmentation**: Elastic & Grid Distortions using Albumentations.
5. **Extensive Visualization**: Verifying every step.

In [ ]:
!pip install albumentations==1.4.0 opencv-python matplotlib scipy tqdm torch torchvision

In [ ]:
import os
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from scipy.spatial.distance import directed_hausdorff

# --- Configuration ---
DATASET_DIR = "/kaggle/input/levir-cd-change-detection/LEVIR-CD+"
PATCH_SIZE = 256
STRIDE = 256  # Non-overlapping for training, can be smaller for inference
BATCH_SIZE = 8
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 1. Visualization Utilities
We define helper functions to visualize triplets and heatmaps immediately.

In [ ]:
def visualize_triplet(img_A, img_B, label, title="Triplet", cmap_label='gray'):
    """Displays Image A, Image B, and the Ground Truth Label side-by-side."""
    plt.figure(figsize=(15, 5))
    plt.suptitle(title, fontsize=16)

    plt.subplot(1, 4, 1)
    plt.imshow(img_A)
    plt.title("Image A (t0)")
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(img_B)
    plt.title("Image B (t1)")
    plt.axis('off')

    # Difference Map Visualization
    diff = np.abs(img_B.astype(np.float32) - img_A.astype(np.float32)).mean(axis=2).astype(np.uint8)
    plt.subplot(1, 4, 3)
    plt.imshow(diff, cmap='hot')
    plt.title("Difference |B - A|")
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(label, cmap=cmap_label)
    plt.title("Ground Truth")
    plt.axis('off')

    plt.show()

def view_tiling_effect(original_A, patches_A, patches_label):
    """Visualizes how an image is broken into patches."""
    print(f"Original Size: {original_A.shape}, Generated Patches: {len(patches_A)}")
    # Show first 5 patches (or fewer)
    n = min(5, len(patches_A))
    plt.figure(figsize=(15, 3))
    plt.suptitle("Tiled Patches (Subset)", fontsize=14)
    for i in range(n):
        plt.subplot(1, n, i+1)
        plt.imshow(patches_A[i])
        plt.title(f"Patch {i}")
        plt.axis('off')
    plt.show()

## 2. Dataset Logic: Tiling & Filtering
We implement the `ChangeDetectionDataset`. 
**Key Feature**: `sliding_window_crop` breaks 1024x1024 images into smaller chunks.
**Key Feature**: `filter_patches` (optional) removes tiles with no change to handle class imbalance.

In [ ]:
class ChangeDetectionDataset(Dataset):
    def __init__(self, root_dir, split='train', patch_size=256, stride=256, transform=None, filter_empty=False):
        self.root_dir = root_dir
        self.split = split
        self.patch_size = patch_size
        self.stride = stride
        self.transform = transform
        self.filter_empty = filter_empty

        self.img_A_dir = os.path.join(root_dir, split, 'A')
        self.img_B_dir = os.path.join(root_dir, split, 'B')
        self.label_dir = os.path.join(root_dir, split, 'label')

        self.filenames = sorted(os.listdir(self.img_A_dir))
        self.samples = self._prepare_samples()
        print(f"[{split}] Found {len(self.filenames)} original images -> {len(self.samples)} patches.")

    def _prepare_samples(self):
        samples = []
        width, height = 1024, 1024 # LEVIR-CD+ dimension config
        
        for fname in self.filenames:
            if not fname.endswith(('.png', '.jpg')):
                continue
            
            for y in range(0, height - self.patch_size + 1, self.stride):
                for x in range(0, width - self.patch_size + 1, self.stride):
                    samples.append((fname, x, y))
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, x, y = self.samples[idx]
        
        path_A = os.path.join(self.img_A_dir, fname)
        path_B = os.path.join(self.img_B_dir, fname)
        path_lbl = os.path.join(self.label_dir, fname)
        
        img_A = cv2.cvtColor(cv2.imread(path_A), cv2.COLOR_BGR2RGB)
        img_B = cv2.cvtColor(cv2.imread(path_B), cv2.COLOR_BGR2RGB)
        label = cv2.imread(path_lbl, cv2.IMREAD_GRAYSCALE)
        
        patch_A = img_A[y:y+self.patch_size, x:x+self.patch_size]
        patch_B = img_B[y:y+self.patch_size, x:x+self.patch_size]
        patch_lbl = label[y:y+self.patch_size, x:x+self.patch_size]
        
        if self.transform:
            augmented = self.transform(image=patch_A, image0=patch_B, mask=patch_lbl)
            patch_A = augmented['image']
            patch_B = augmented['image0']
            patch_lbl = augmented['mask']
            
        return patch_A, patch_B, patch_lbl

## 3 Data Normalization

In [ ]:
def get_transforms(phase='train'):
    if phase == 'train':
        return A.Compose([
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ], additional_targets={'image0': 'image'})
    else:
        return A.Compose([
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ], additional_targets={'image0': 'image'})

## 4. Visualization Checkpoints
Run these cells to verify data before training.

In [ ]:
# VISUALIZATION 1: Raw Patches
viz_dataset = ChangeDetectionDataset(DATASET_DIR, split='train', transform=None)

print("Visualizing Random Raw Patches...")
indices = random.sample(range(len(viz_dataset)), 3)
for idx in indices:
    img_A, img_B, label = viz_dataset[idx]
    visualize_triplet(img_A, img_B, label, title=f"Sample Patch {idx}")

## 5. Utility Functions
Comprehensive utilities for metrics, visualization, and training management.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import seaborn as sns

def calculate_metrics(predictions, targets, threshold=0.5):
    """Calculate evaluation metrics for change detection."""
    if predictions.max() > 1 or predictions.min() < 0:
        preds = torch.sigmoid(predictions)
    else:
        preds = predictions
    
    preds_binary = (preds > threshold).float()
    preds_flat = preds_binary.view(-1).cpu().numpy()
    
    # Binarize targets as well (in case they're continuous 0-1)
    targets_binary = (targets > 0.5).float()
    targets_flat = targets_binary.view(-1).cpu().numpy()
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        targets_flat, preds_flat, average='binary', zero_division=0
    )
    accuracy = (preds_flat == targets_flat).mean()
    intersection = (preds_flat * targets_flat).sum()
    union = preds_flat.sum() + targets_flat.sum() - intersection
    iou = intersection / (union + 1e-6)
    
    return {'precision': precision, 'recall': recall, 'f1_score': f1, 'accuracy': accuracy, 'iou': iou}

def plot_training_history(history, save_path=None):
    """Plot training and validation curves."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    axes[0].plot(history['train_loss'], label='Train', linewidth=2, color='#1f77b4')
    axes[0].plot(history['val_loss'], label='Val', linewidth=2, color='#ff7f0e')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(history['val_f1_score'], label='F1', linewidth=2, color='#2ca02c')
    axes[1].plot(history['val_iou'], label='IoU', linewidth=2, color='#d62728')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
    axes[1].set_title('F1 & IoU', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[2].plot(history['val_precision'], label='Precision', linewidth=2, color='#9467bd')
    axes[2].plot(history['val_recall'], label='Recall', linewidth=2, color='#8c564b')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Score')
    axes[2].set_title('Precision & Recall', fontweight='bold'); axes[2].legend(); axes[2].grid(alpha=0.3)
    axes[3].plot(history['val_accuracy'], label='Accuracy', linewidth=2, color='#e377c2')
    axes[3].set_xlabel('Epoch'); axes[3].set_ylabel('Accuracy')
    axes[3].set_title('Accuracy', fontweight='bold'); axes[3].legend(); axes[3].grid(alpha=0.3)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

print("Metrics & visualization utilities loaded!")

In [ ]:
def visualize_predictions(images_a, images_b, labels, predictions, num_samples=4, save_path=None):
    """Enhanced 8-column visualization."""
    num_samples = min(num_samples, images_a.shape[0])
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(images_a.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(images_a.device)
    images_a_denorm = images_a * std + mean
    images_b_denorm = images_b * std + mean
    pred_probs = torch.sigmoid(predictions)
    pred_binary = (pred_probs > 0.5).float()
    
    fig, axes = plt.subplots(num_samples, 8, figsize=(32, 4 * num_samples))
    if num_samples == 1: axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        img_a = np.clip(images_a_denorm[i].permute(1, 2, 0).cpu().numpy(), 0, 1)
        img_b = np.clip(images_b_denorm[i].permute(1, 2, 0).cpu().numpy(), 0, 1)
        axes[i, 0].imshow(img_a); axes[i, 0].set_title('Image A (t)', fontweight='bold'); axes[i, 0].axis('off')
        axes[i, 1].imshow(img_b); axes[i, 1].set_title('Image B (t+1)', fontweight='bold'); axes[i, 1].axis('off')
        
        diff_gray = np.mean(np.abs(img_b - img_a), axis=2)
        im = axes[i, 2].imshow(diff_gray, cmap='hot', vmin=0, vmax=0.5)
        axes[i, 2].set_title('Diff |B-A|', fontweight='bold'); axes[i, 2].axis('off')
        plt.colorbar(im, ax=axes[i, 2], fraction=0.046)
        
        gt = labels[i, 0].cpu().numpy()
        axes[i, 3].imshow(gt, cmap='Reds', vmin=0, vmax=1)
        axes[i, 3].set_title('Ground Truth', fontweight='bold'); axes[i, 3].axis('off')
        
        pred_prob = pred_probs[i, 0].detach().cpu().numpy()
        im = axes[i, 4].imshow(pred_prob, cmap='jet', vmin=0, vmax=1)
        axes[i, 4].set_title('Pred (Prob)', fontweight='bold'); axes[i, 4].axis('off')
        plt.colorbar(im, ax=axes[i, 4], fraction=0.046)
        
        pred_bin = pred_binary[i, 0].detach().cpu().numpy()
        axes[i, 5].imshow(pred_bin, cmap='Reds', vmin=0, vmax=1)
        axes[i, 5].set_title('Pred (Binary)', fontweight='bold'); axes[i, 5].axis('off')
        
        error_map = np.zeros((gt.shape[0], gt.shape[1], 3))
        error_map[(gt == 1) & (pred_bin == 1), 1] = 1.0  # TP=Green
        error_map[(gt == 0) & (pred_bin == 1), 0] = 1.0  # FP=Red
        error_map[(gt == 1) & (pred_bin == 0), 2] = 1.0  # FN=Blue
        axes[i, 6].imshow(error_map)
        axes[i, 6].set_title('Error\n(G=TP,R=FP,B=FN)', fontweight='bold'); axes[i, 6].axis('off')
        
        overlay = img_b.copy()
        overlay[pred_bin == 1, 0] = np.minimum(overlay[pred_bin == 1, 0] + 0.4, 1.0)
        axes[i, 7].imshow(overlay); axes[i, 7].set_title('Overlay', fontweight='bold'); axes[i, 7].axis('off')
    
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

def plot_confusion_matrix(predictions, targets, save_path=None):
    """Plot confusion matrix."""
    preds_binary = (torch.sigmoid(predictions) > 0.5).float()
    targets_binary = (targets > 0.5).float()
    cm = confusion_matrix(targets_binary.view(-1).cpu().numpy(), preds_binary.view(-1).cpu().numpy())
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Change', 'Change'], yticklabels=['No Change', 'Change'])
    plt.xlabel('Predicted', fontweight='bold'); plt.ylabel('Actual', fontweight='bold')
    plt.title('Confusion Matrix', fontweight='bold')
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

print("Prediction visualization utilities loaded!")

In [ ]:
class EarlyStopping:
    """Early stopping to prevent overfitting."""
    def __init__(self, patience=10, mode='max', min_delta=0):
        self.patience, self.mode, self.min_delta = patience, mode, min_delta
        self.counter, self.best_score, self.early_stop = 0, None, False
    
    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return True
        improved = (self.mode == 'max' and score > self.best_score + self.min_delta) or \
                   (self.mode == 'min' and score < self.best_score - self.min_delta)
        if improved:
            self.best_score, self.counter = score, 0
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.early_stop = True
        return False

def save_checkpoint(model, optimizer, epoch, metrics, filepath):
    """Save model checkpoint."""
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(), 'metrics': metrics}, 
               filepath)
    print(f"Checkpoint saved: {filepath}")

def load_checkpoint(model, optimizer, filepath, device):
    """Load model checkpoint."""
    checkpoint = torch.load(filepath, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer: optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print(f"Checkpoint loaded: {filepath}")
    return checkpoint['epoch'], checkpoint['metrics']

print("Training utilities loaded!")

# Phase 2: Classical Model Architecture

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=6, out_channels=1, init_features=32):
        super(UNet, self).__init__()
        features = init_features
        self.encoder1 = UNet._block(in_channels, features)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder2 = UNet._block(features, features * 2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder3 = UNet._block(features * 2, features * 4)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder4 = UNet._block(features * 4, features * 8)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = UNet._block(features * 8, features * 16)
        self.upconv4 = nn.ConvTranspose2d(features * 16, features * 8, kernel_size=2, stride=2)
        self.decoder4 = UNet._block((features * 8) * 2, features * 8)
        self.upconv3 = nn.ConvTranspose2d(features * 8, features * 4, kernel_size=2, stride=2)
        self.decoder3 = UNet._block((features * 4) * 2, features * 4)
        self.upconv2 = nn.ConvTranspose2d(features * 4, features * 2, kernel_size=2, stride=2)
        self.decoder2 = UNet._block((features * 2) * 2, features * 2)
        self.upconv1 = nn.ConvTranspose2d(features * 2, features, kernel_size=2, stride=2)
        self.decoder1 = UNet._block(features * 2, features)
        self.conv = nn.Conv2d(in_channels=features, out_channels=out_channels, kernel_size=1)

    @staticmethod
    def _block(in_channels, features, name='block'):
        return nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=features, out_channels=features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, y=None):
        if y is not None:
            x = torch.cat([x, y], dim=1)
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        enc3 = self.encoder3(self.pool2(enc2))
        enc4 = self.encoder4(self.pool3(enc3))
        bottleneck = self.bottleneck(self.pool4(enc4))
        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4(dec4)
        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        return self.conv(dec1)

model = UNet().to(DEVICE)
# Check for forward pass
dummy = torch.randn(2, 3, PATCH_SIZE, PATCH_SIZE).to(DEVICE)
out = model(dummy, dummy)
print(f"Output Shape: {out.shape}")

In [ ]:
from torchinfo import summary
summary(
    model,
    input_size=[(1,3,256, 256), (1,3,256, 256)]
)

# Phase 3 & 4: Training & Visualization
Implementing **Hybrid Loss** and the Training Loop with **Deep Supervision**.

## Enhanced Training Loop
Production-ready training with mixed precision, gradient accumulation, and comprehensive evaluation.

In [ ]:
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR

# Enhanced Training Configuration
NUM_EPOCHS = 20
GRADIENT_ACCUMULATION_STEPS = 4
GRADIENT_CLIP = 2.0
USE_MIXED_PRECISION = True
MIN_LR = 1e-6
SAVE_DIR = "checkpoints"
RESULTS_DIR = "results"
import os
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Training Configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Device: {DEVICE}")

In [ ]:
from torch.utils.data import random_split

# Create full dataset
full_dataset = ChangeDetectionDataset(DATASET_DIR, split='train', transform=get_transforms('train'))

# Calculate split sizes (70/20/10)
total_size = len(full_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size

# Split dataset
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dataset Split (70/20/10):")
print(f"  Training: {len(train_dataset)} samples ({len(train_dataset)/total_size*100:.1f}%)")
print(f"  Validation: {len(val_dataset)} samples ({len(val_dataset)/total_size*100:.1f}%)")
print(f"  Test: {len(test_dataset)} samples ({len(test_dataset)/total_size*100:.1f}%)")
print(f"  Total: {total_size} samples")
print(f"\nDevice: {DEVICE}")

In [ ]:
def simple_visualize(images_a, images_b, labels, predictions, num_samples=4, save_path=None):
    """Simple 4-column visualization: t1, t2, ground truth, prediction."""
    num_samples = min(num_samples, images_a.shape[0])
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(images_a.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(images_a.device)
    images_a_denorm = images_a * std + mean
    images_b_denorm = images_b * std + mean
    pred_binary = (torch.sigmoid(predictions) > 0.5).float()
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    if num_samples == 1: axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        img_a = np.clip(images_a_denorm[i].permute(1, 2, 0).cpu().numpy(), 0, 1)
        img_b = np.clip(images_b_denorm[i].permute(1, 2, 0).cpu().numpy(), 0, 1)
        gt = labels[i, 0].cpu().numpy()
        pred = pred_binary[i, 0].detach().cpu().numpy()
        
        axes[i, 0].imshow(img_a)
        axes[i, 0].set_title('Image at t1', fontweight='bold', fontsize=12)
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(img_b)
        axes[i, 1].set_title('Image at t2', fontweight='bold', fontsize=12)
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(gt, cmap='Reds', vmin=0, vmax=1)
        axes[i, 2].set_title('Ground Truth', fontweight='bold', fontsize=12)
        axes[i, 2].axis('off')
        
        axes[i, 3].imshow(pred, cmap='Reds', vmin=0, vmax=1)
        axes[i, 3].set_title('Prediction', fontweight='bold', fontsize=12)
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

print("Simple visualization function loaded!")

In [ ]:
class DiceLoss(nn.Module):
    """Dice Loss for binary segmentation."""
    def __init__(self, smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice = (2.*intersection + self.smooth)/(inputs.sum() + targets.sum() + self.smooth)
        return 1 - dice

print("DiceLoss defined!")

In [ ]:
# Setup training components
criterion_dice = DiceLoss()
criterion_bce = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=MIN_LR)
scaler = GradScaler() if USE_MIXED_PRECISION else None
early_stopping = EarlyStopping(patience=10, mode='max', min_delta=0.001)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_precision': [],
    'val_recall': [],
    'val_f1_score': [],
    'val_iou': [],
    'val_accuracy': []
}

best_f1 = 0

print("Training components initialized!")
print(f"Mixed Precision: {USE_MIXED_PRECISION}")
print(f"Gradient Accumulation Steps: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Gradient Clipping: {GRADIENT_CLIP}")

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion_dice, criterion_bce, scaler, epoch):
    """Train for one epoch with gradient accumulation and mixed precision."""
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    
    loop = tqdm(loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for batch_idx, (A, B, label) in enumerate(loop):
        A, B, label = A.to(DEVICE), B.to(DEVICE), label.float().unsqueeze(1).to(DEVICE) / 255.0
        
        # Forward with mixed precision
        if scaler:
            with autocast():
                outputs = model(torch.cat([A, B], dim=1))
                loss = (criterion_dice(outputs, label) + criterion_bce(outputs, label)) / GRADIENT_ACCUMULATION_STEPS
            scaler.scale(loss).backward()
        else:
            outputs = model(torch.cat([A, B], dim=1))
            loss = (criterion_dice(outputs, label) + criterion_bce(outputs, label)) / GRADIENT_ACCUMULATION_STEPS
            loss.backward()
        
        # Gradient accumulation step
        if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            if GRADIENT_CLIP > 0:
                if scaler:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            
            if scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()
        
        running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        loop.set_postfix(loss=f'{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}')
    
    return running_loss / len(loader)

def validate(model, loader, criterion_dice, criterion_bce, epoch):
    """Validate and return metrics."""
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    sample_a, sample_b, sample_gt, sample_pred = None, None, None, None
    
    with torch.no_grad():
        for batch_idx, (A, B, label) in enumerate(tqdm(loader, desc='Validation')):
            A, B, label = A.to(DEVICE), B.to(DEVICE), label.float().unsqueeze(1).to(DEVICE) / 255.0
            outputs = model(torch.cat([A, B], dim=1))
            loss = criterion_dice(outputs, label) + criterion_bce(outputs, label)
            running_loss += loss.item()
            all_preds.append(outputs)
            all_labels.append(label)
            
            if batch_idx == 0:
                sample_a, sample_b, sample_gt, sample_pred = A, B, label, outputs
    
    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    metrics = calculate_metrics(all_preds, all_labels)
    
    # Visualize every 5 epochs
    if (epoch + 1) % 5 == 0:
        viz_path = f'{RESULTS_DIR}/predictions_epoch_{epoch+1}.png'
        simple_visualize(sample_a, sample_b, sample_gt, sample_pred, num_samples=4, save_path=viz_path)
    
    return running_loss / len(loader), metrics

print("Training and validation functions defined!")

In [ ]:
# Main training loop
print("\n" + "="*60)
print("Starting Enhanced Training")
print("="*60)

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion_dice, criterion_bce, scaler, epoch)
    
    # Validate
    val_loss, val_metrics = validate(model, val_loader, criterion_dice, criterion_bce, epoch)
    
    # Update scheduler
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    for key in ['precision', 'recall', 'f1_score', 'iou', 'accuracy']:
        history[f'val_{key}'].append(val_metrics[key])
    
    # Print metrics
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"  F1: {val_metrics['f1_score']:.4f} | IoU: {val_metrics['iou']:.4f} | Acc: {val_metrics['accuracy']:.4f}")
    
    # Save best model
    if val_metrics['f1_score'] > best_f1:
        best_f1 = val_metrics['f1_score']
        save_checkpoint(model, optimizer, epoch, val_metrics, f'{SAVE_DIR}/best_model.pth')
        print(f"  ✓ Best model saved (F1: {best_f1:.4f})")
    
    # Save latest
    save_checkpoint(model, optimizer, epoch, val_metrics, f'{SAVE_DIR}/latest_model.pth')
    
    # Early stopping
    if not early_stopping(val_metrics['f1_score']):
        if early_stopping.early_stop:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print("\nTraining completed!")

In [ ]:
# Plot training history
plot_training_history(history, save_path=f'{RESULTS_DIR}/training_history.png')

## Final Evaluation
Comprehensive evaluation on the validation set with the best model.

In [ ]:
# Load best model
print("Loading best model for final evaluation...")
load_checkpoint(model, None, f'{SAVE_DIR}/best_model.pth', DEVICE)

# Final evaluation
model.eval()
all_preds, all_labels = [], []
sample_a, sample_b, sample_gt, sample_pred = None, None, None, None

with torch.no_grad():
    for batch_idx, (A, B, label) in enumerate(tqdm(val_loader, desc='Final Evaluation')):
        A, B, label = A.to(DEVICE), B.to(DEVICE), label.float().unsqueeze(1).to(DEVICE) / 255.0
        outputs = model(torch.cat([A, B], dim=1))
        all_preds.append(outputs)
        all_labels.append(label)
        
        if batch_idx == 0:
            sample_a, sample_b, sample_gt, sample_pred = A, B, label, outputs

all_preds = torch.cat(all_preds, dim=0)
all_labels = torch.cat(all_labels, dim=0)

# Calculate final metrics
final_metrics = calculate_metrics(all_preds, all_labels)

print("\n" + "="*60)
print("Final Metrics on Validation Set")
print("="*60)
for key, value in final_metrics.items():
    print(f"  {key.capitalize()}: {value:.4f}")
print("="*60)

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(all_preds, all_labels, save_path=f'{RESULTS_DIR}/confusion_matrix.png')

In [ ]:
# Simple 4-column visualization
print("\nGenerating final predictions visualization...")
simple_visualize(sample_a, sample_b, sample_gt, sample_pred, num_samples=6, save_path=f'{RESULTS_DIR}/final_simple_predictions.png')

In [ ]:
# Enhanced 8-column visualization
print("\nGenerating enhanced predictions visualization...")
visualize_predictions(sample_a, sample_b, sample_gt, sample_pred, num_samples=4, save_path=f'{RESULTS_DIR}/final_enhanced_predictions.png')

## Test Set Evaluation
Final evaluation on the held-out test set (10% of data).

In [ ]:
# Load best model for test evaluation
print("Loading best model for test set evaluation...")
load_checkpoint(model, None, f'{SAVE_DIR}/best_model.pth', DEVICE)

# Test evaluation
model.eval()
test_preds, test_labels = [], []
test_sample_a, test_sample_b, test_sample_gt, test_sample_pred = None, None, None, None

print("\nEvaluating on test set...")
with torch.no_grad():
    for batch_idx, (A, B, label) in enumerate(tqdm(test_loader, desc='Test Evaluation')):
        A, B, label = A.to(DEVICE), B.to(DEVICE), label.float().unsqueeze(1).to(DEVICE) / 255.0
        outputs = model(torch.cat([A, B], dim=1))
        test_preds.append(outputs)
        test_labels.append(label)
        
        if batch_idx == 0:
            test_sample_a, test_sample_b, test_sample_gt, test_sample_pred = A, B, label, outputs

test_preds = torch.cat(test_preds, dim=0)
test_labels = torch.cat(test_labels, dim=0)

# Calculate test metrics
test_metrics = calculate_metrics(test_preds, test_labels)

print("\n" + "="*60)
print("FINAL TEST SET METRICS")
print("="*60)
for key, value in test_metrics.items():
    print(f"  {key.upper()}: {value:.4f}")
print("="*60)

In [ ]:
# Test set confusion matrix
print("\nGenerating test set confusion matrix...")
plot_confusion_matrix(test_preds, test_labels, save_path=f'{RESULTS_DIR}/test_confusion_matrix.png')

In [ ]:
# Test set simple 4-column visualization
print("\nGenerating test set predictions (simple view)...")
simple_visualize(
    test_sample_a, test_sample_b, test_sample_gt, test_sample_pred,
    num_samples=6, save_path=f'{RESULTS_DIR}/test_simple_predictions.png'
)

In [ ]:
# Test set enhanced 8-column visualization
print("\nGenerating test set predictions (enhanced view)...")
visualize_predictions(
    test_sample_a, test_sample_b, test_sample_gt, test_sample_pred,
    num_samples=4, save_path=f'{RESULTS_DIR}/test_enhanced_predictions.png'
)

# Phase 5: Evaluation & Uncertainty
**MC Dropout** for Uncertainty Mapping and **Metric Calculation**.

In [ ]:
def calculate_uncertainty(model, img_A, img_B, T=10):
    """Monte Carlo Dropout Inference."""
    model.train() # Enable Dropout
    preds = []
    with torch.no_grad():
        for _ in range(T):
            out = torch.sigmoid(model(img_A.unsqueeze(0), img_B.unsqueeze(0)))
            preds.append(out.cpu().numpy())
    
    preds = np.array(preds).squeeze()
    mean_pred = preds.mean(axis=0)
    variance = preds.var(axis=0)
    return mean_pred, variance

# Test on a random sample
print("Running MC Dropout Uncertainty Test...")
idx = random.choice(range(len(viz_dataset)))
img_A, img_B, label = viz_dataset[idx]
# Convert to Tensor
t_A = ToTensorV2()(image=img_A)['image'].float().to(DEVICE)
t_B = ToTensorV2()(image=img_B)['image'].float().to(DEVICE)

mean, var = calculate_uncertainty(model, t_A, t_B)

plt.figure(figsize=(10, 4))
plt.subplot(1, 3, 1); plt.imshow(mean, cmap='jet'); plt.title("Mean Prediction")
plt.subplot(1, 3, 2); plt.imshow(var, cmap='plasma'); plt.title("Uncertainty (Variance)")
plt.subplot(1, 3, 3); plt.imshow(label, cmap='gray'); plt.title("Ground Truth")
plt.show()